# Quantum Support Vector Machines in 4 Leptons Data Set - Atlas Open Data

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import uproot
import matplotlib.text
import sys
import awkward as ak
from atlasopenmagic import install_from_environment
import atlasopenmagic as atom
import qiskit
from qiskit_machine_learning.datasets import ad_hoc_data
from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import minmax_scale
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

In [37]:
atom.set_release('2025e-13tev-beta')

Release '2025e-13tev-beta' already active with cached metadata.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [3]:
atom.available_skims()

['1LMET30',
 '2J2LMET30',
 '2bjets',
 '2muons',
 '2to4lep',
 '3J1LMET30',
 '3lep',
 '4lep',
 'GamGam',
 'exactly3lep',
 'exactly4lep']

In [77]:
skim='GamGam'
signal_list = atom.get_urls('346189', skim=skim, protocol= 'https', cache = True) #This is our pp->ttbarH->gamgam
bkg_GGFH = atom.get_urls('345944',skim=skim, protocol= 'https', cache = True)  #GGF H->gamma gamma also produces a higgs so it is similar but not the same
bkg_qqZH = atom.get_urls('341456',skim=skim, protocol= 'https', cache = True) #qq->ZH, H->WW, Z->veve
background_lists =  bkg_GGFH +bkg_qqZH          

In [ ]:
print('signal:', signal_list) #29592 events
print('background:', background_lists)

['simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_346189.aMcAtNloPythia8EvtGen_ttH_gamgam.GamGam.root']


Going by Wu, et al. the features tha I'll be using will be the following

| Name of Feature   | Description |
| -------- | ------- |
| photon_pt  | The transverse momentum of the photons. <br> Since we analizing events with only two photons, <br> there are only 2 elements to this feature. <br> It is important to distinguish the photons by sorting <br> them according to their transverse momentum. <br> As such calling the photon with the highest transverse <br> momentum leading photon and the other one subleading   |
| photon_eta | pseudorapidity of our photons     |
| jet_pt    | Transverse momentum of our jets, here we can expect at most 6 jets <br> it is important to clarify that we may also end up <br> in an event with few or 0 jets |
| jet_n | pseudorapidity of our jets    |
| jet_btag_quantile |    did the jet at which we are looking to come from bottom quarks  |
| met    | missing transverse momentum, that is the energy that is missing from what we <br> would have awaited to obtain    |




In [84]:
attributes = ['photon_pt', 'photon_eta', 'jet_pt', 'jet_eta','jet_btag_quantile', 'met']
aliases_dict = {'photon_pt': 'photon_pt', 'photon_eta': 'photon_eta', 'jet_pt': 'jet_pt', ' jet_eta': 'jet_eta',
                  'btag': 'jet_btag_quantile', 'met': 'met'}

In [75]:
data = []
for file in background_lists + signal_list:
    print(file)
    analysis = uproot.open(file + ':analysis')
    attributes = ['photon_pt', 'photon_eta', 'jet_pt', 'jet_eta','jet_btag_quantile', 'met']
    aliases_dict = {'photon_pt': 'photon_pt', 'photon_eta': 'photon_eta', 'jet_pt': 'jet_pt', ' jet_eta': 'jet_eta',
                    'btag': 'jet_btag_quantile', 'met': 'met'}
    partial_data = analysis.arrays(attributes , aliases = aliases_dict) ## casi me olvido del .arrays
    data=np.concatenate([data, partial_data] , axis = 0)
print(data)

simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_345944.aMcAtNloPy8EG_A14NNPDF23LO_ppx0yy_FxFx_Np012_SM.GamGam.root
simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_341456.PowhegPythia8EvtGen_CT10_AZNLO_ZH125J_MINLO_veveWWlvqq_VpT.GamGam.root
simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/opendata/ODEO_FEB2025_v0_GamGam_mc_346189.aMcAtNloPythia8EvtGen_ttH_gamgam.GamGam.root
[{photon_pt: [40, 34.8], photon_eta: [-2.22, ...], jet_pt: [], ...}, ..., {...}]


In [102]:
data.fields

['photon_pt', 'photon_eta', 'jet_pt', 'jet_eta', 'jet_btag_quantile', 'met']

In [ ]:
labels = np.concatenate([np.zeros(123573-29592),np.ones(29592)]) #0 for background, 1 for signal
data_with_labels = ak.with_field(data, labels, 'labels' ) 

In [137]:
ak.num(data.met,axis= 0)

array(123573)

In [139]:
data.met.type


ArrayType(NumpyType('float32'), 123573, None)

In [140]:
data.jet_pt.type

ArrayType(ListType(NumpyType('float32')), 123573, None)

In [132]:
len(data.met)

123573

In [ ]:
ak.max(ak.num(data_with_labels.met,axis= 1))

AxisError: axis=1 exceeds the depth of this array (1)

In [118]:
for f in data_with_labels.fields:
    print(f ,':', ak.max(ak.num(data_with_labels[f],axis= 1)))

photon_pt : 5
photon_eta : 5
jet_pt : 14
jet_eta : 14
jet_btag_quantile : 14


AxisError: axis=1 exceeds the depth of this array (1)

In [101]:
sorter = np.argsort(data.photon_pt , axis = 1)[:,::-1]
data = data[sorter]

IndexError: cannot slice ListArray (of length 123573) with [[0, 1], [0, 1], [1, 0], [0, 1], [0, ...], ..., [0, 1], [0, 1], [0, 1], [0, 1]]: index out of range while attempting to get index 0 (in compiled code: https://github.com/scikit-hep/awkward/blob/awkward-cpp-56/awkward-cpp/src/cpu-kernels/awkward_ListArray_getitem_jagged_apply.cpp#L43)

In [ ]:
photons_and_jets = ak.to_numpy()

In [35]:
N = 4
vecs= None
for f in photons_and_jets.fields: 
    
    a = photons_and_jets[f]

    if a.ndim > 1:
        a = ak.fill_none(ak.pad_none(a,N,clip=True),0)
        if vecs is None:
            vecs = ak.to_numpy(a).reshape(len(a),-1)
        else:
            vecs = np.concatenate((vecs, ak.to_numpy(a).reshape(len(a),-1)),axis = 1)
    else:
       vecs=np.concatenate((vecs, ak.to_numpy(a).reshape(len(a),1)), axis = 1)

In [15]:
vecs.shape

(63195, 21)

In [27]:
vecs_scaled = minmax_scale(vecs,(0,np.pi))

In [23]:
backend = BasicSimulator()
backend.set_options(
    shots = 1024,
    seed_simulator = 42
)

In [33]:
feature_map = zz_feature_map(
    feature_dimension=21,
    reps= 10,
    entanglement= "circular"
)


In [31]:
kernel = FidelityQuantumKernel(
    feature_map=feature_map
)